In [3]:
#I'm writing comments for the rabbit Leap problem for future reference

#Rabbit leap problem
class State:
     
    # goal state
    GOAL_STATE = ('W', 'W', 'W', '_', 'E', 'E', 'E')
    

    def __init__(self, board):
        
        self.board = tuple(board)
        self.num_stones = len(self.board)

    def goalTest(self):
        return self.board == self.GOAL_STATE

    def moveGen(self):

        # Generates all possible next states (children) from the current state.
        # This method is required by the search functions.
        
        # A rabbit can:
        #1. Slide forward into an adjacent empty space.
        #2. Jump forward over one other rabbit into an empty space.
        
        #Returns: list[State]: A list of new State objects representing valid moves.
        
        children = []

        # findig the index of the empty space ('_')
        # using try-except to handle the case where no empty space is found
        try:
            empty_idx = self.board.index('_')
        except ValueError:
            # This case should not happen in a valid puzzle state
            return []

        
        # 1. East-bound rabbit ('E') moves to the right
        # Slide (E _) -> (_ E)
        if empty_idx > 0 and self.board[empty_idx - 1] == 'E':
            new_board_list = list(self.board)
            new_board_list[empty_idx], new_board_list[empty_idx - 1] = new_board_list[empty_idx - 1], new_board_list[empty_idx]
            children.append(State(tuple(new_board_list)))
        # Jump (E W _) -> (_ W E)
        if empty_idx > 1 and self.board[empty_idx - 2] == 'E':
            new_board_list = list(self.board)
            new_board_list[empty_idx], new_board_list[empty_idx - 2] = new_board_list[empty_idx - 2], new_board_list[empty_idx]
            children.append(State(tuple(new_board_list)))

        # 2. West-bound rabbit ('W') moves to the left
        # Slide (_ W) -> (W _)
        if empty_idx < self.num_stones - 1 and self.board[empty_idx + 1] == 'W':
            new_board_list = list(self.board)
            new_board_list[empty_idx], new_board_list[empty_idx + 1] = new_board_list[empty_idx + 1], new_board_list[empty_idx]
            children.append(State(tuple(new_board_list)))
        # Jump (_ E W) -> (W E _)
        if empty_idx < self.num_stones - 2 and self.board[empty_idx + 2] == 'W':
            new_board_list = list(self.board)
            new_board_list[empty_idx], new_board_list[empty_idx + 2] = new_board_list[empty_idx + 2], new_board_list[empty_idx]
            children.append(State(tuple(new_board_list)))
            
        return children

    # --- Helper methods for printing and comparison ---
    def __str__(self):
        """String representation for printing the path."""
        return " ".join(self.board)
        
    def __repr__(self):
        """String representation for debugging."""
        return f"State({self.board})"

    def __eq__(self, other):
        """Checks if two State objects are equal based on their board."""
        return isinstance(other, State) and self.board == other.board

    def __hash__(self):
        """Makes the State object hashable for use in sets and dictionaries."""
        return hash(self.board)


In [4]:
#auxillary functions
def reconstructPath(node_pair, closed_list):
    """
    Traces back from the goal node to the start node to build the solution path.
    """
    path = []
    current_node, parent_node = node_pair
    path.append(current_node)
    
    # Create a dictionary for efficient parent lookups from the CLOSED list
    parent_map = {node: parent for node, parent in closed_list}
    parent_map[current_node] = parent_node # Add the final node's parent

    while parent_node is not None:
        path.append(parent_node)
        current_node = parent_node
        parent_node = parent_map.get(current_node)
        
    return path

def removeSeen(children, open_list, closed_list):

    #Filters out children that have already been seen (in OPEN or CLOSED lists).

    seen_in_open = {node for node, parent in open_list}
    seen_in_closed = {node for node, parent in closed_list}
    
    new_nodes = [child for child in children if child not in seen_in_open and child not in seen_in_closed]
            
    return new_nodes


def bfs(start):
 
    OPEN = [(start, None)]
    CLOSED  = []
    while OPEN:
        node_pair = OPEN.pop(0)
        N, parent = node_pair
        
        if N.goalTest():
            print("Goal found using BFS!")
            path = reconstructPath(node_pair, CLOSED)
            path.reverse()
            
            for i, p in enumerate(path):
                print(f"Step {i:>2}: {p}")
            
            return 
        else:
            CLOSED.append(node_pair)
            children = N.moveGen()
            new_nodes = removeSeen(children, OPEN, CLOSED)
            new_pairs = [(c, N) for c in new_nodes]
            OPEN.extend(new_pairs) 
    
    print("No solution found with BFS.")
    return []


def dfs(start):
     
    OPEN = [(start, None)]
    CLOSED  = []
    while OPEN:
        node_pair = OPEN.pop(0)
        N, parent = node_pair
        
        if N.goalTest():
            print("Goal found using DFS!")
            path = reconstructPath(node_pair, CLOSED)
            path.reverse()
            
            for i, p in enumerate(path):
                print(f"Step {i:>2}: {p}")
            
            return 
        else:
            CLOSED.append(node_pair)
            children = N.moveGen()
            new_nodes = removeSeen(children, OPEN, CLOSED)
            new_pairs = [(c, N) for c in new_nodes]
            OPEN = new_pairs + OPEN

    print("No solution found with DFS.")
    return []

  
start_node = State(('E', 'E', 'E', '_', 'W', 'W', 'W'))

print("--- Running Breadth-First Search (BFS) ---")
print("BFS guarantees finding the shortest path.\n")
bfs(start_node)
print("\n")
print("--- Running Depth-First Search (DFS) ---")
print("DFS  may or may not guarantee int finding the shortest path.\n")
dfs(start_node)

--- Running Breadth-First Search (BFS) ---
BFS guarantees finding the shortest path.

Goal found using BFS!
Step  0: E E E _ W W W
Step  1: E E _ E W W W
Step  2: E E W E _ W W
Step  3: E E W E W _ W
Step  4: E E W _ W E W
Step  5: E _ W E W E W
Step  6: _ E W E W E W
Step  7: W E _ E W E W
Step  8: W E W E _ E W
Step  9: W E W E W E _
Step 10: W E W E W _ E
Step 11: W E W _ W E E
Step 12: W _ W E W E E
Step 13: W W _ E W E E
Step 14: W W W E _ E E
Step 15: W W W _ E E E


--- Running Depth-First Search (DFS) ---
DFS  may or may not guarantee int finding the shortest path.

Goal found using DFS!
Step  0: E E E _ W W W
Step  1: E E _ E W W W
Step  2: E E W E _ W W
Step  3: E E W E W _ W
Step  4: E E W _ W E W
Step  5: E _ W E W E W
Step  6: _ E W E W E W
Step  7: W E _ E W E W
Step  8: W E W E _ E W
Step  9: W E W E W E _
Step 10: W E W E W _ E
Step 11: W E W _ W E E
Step 12: W _ W E W E E
Step 13: W W _ E W E E
Step 14: W W W E _ E E
Step 15: W W W _ E E E
